# KIMIDORI Movie Auto - Google Colab 無料T4 GPU API ワーカー

Google Colabの無料T4 GPU（16GB VRAM）を活用し、完全無料で **FLUX.1 [schnell]** による画像生成および **CogVideoX / AnimateDiff** による動画生成を行うためのAPIバックエンドです。

### 【利用手順】
1. メニューの **「ランタイム」→「ランタイムのタイプを変更」** で **「T4 GPU」** が選択されていることを確認します。
2. 以下のセルを上から順番に（または「すべてのセルを実行」で）実行します。
3. 発行された `https://xxxx.trycloudflare.com` のURLをコピーし、KIMIDORI Webアプリの「API・連携設定」画面の **「Colab GPUワーカーURL」** に貼り付けます。

In [ ]:
# Step 1: 必要なライブラリのインストール
!pip install -q fastapi uvicorn pyngrok diffusers transformers accelerate torch sentencepiece
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("Libraries installed successfully!")

In [ ]:
# Step 2: GPUワーカーサーバーの起動と公開URLの発行
import io, os, base64, torch, subprocess, threading
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn

app = FastAPI(title="KIMIDORI Colab Free GPU Worker")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

device = "cuda" if torch.cuda.is_available() else "cpu"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1) if torch.cuda.is_available() else 0
print(f"Device: {device}, VRAM: {vram_gb} GB")

flux_pipe = None
def init_flux():
    global flux_pipe
    if flux_pipe is None and torch.cuda.is_available():
        from diffusers import FluxPipeline
        flux_pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16)
        flux_pipe.enable_model_cpu_offload()
        print("FLUX.1 [schnell] loaded successfully!")

class ImageGenRequest(BaseModel):
    prompt: str
    width: Optional[int] = 1024
    height: Optional[int] = 576
    steps: Optional[int] = 4

@app.get("/health")
def health():
    return {"status": "healthy", "gpu_available": torch.cuda.is_available(), "vram_gb": vram_gb}

@app.post("/generate/image")
def gen_image(req: ImageGenRequest):
    global flux_pipe
    if flux_pipe is None: init_flux()
    img = flux_pipe(prompt=req.prompt, width=req.width, height=req.height, num_inference_steps=req.steps, guidance_scale=0.0).images[0]
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return {"status": "success", "image_base64": f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"}

# バックグラウンドでUvicorn起動
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000), daemon=True).start()

# Cloudflare Tunnelの起動
p = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "trycloudflare.com" in line:
        for part in line.split():
            if "trycloudflare.com" in part:
                print("\n=======================================================")
                print(f"★ KIMIDORI Colab Worker URL: {part}")
                print("=======================================================\n")
                break
        break
